In [71]:
"""
Hall & Jones (1999) Replication — Analysis Script
==================================================
Produces:
  Table I   : Levels accounting decomposition
  Table II  : IV regression (OLS + 2SLS, robust SEs)
  Figure I  : Y/L vs social infrastructure scatter
  Figure II : TFP vs social infrastructure scatter
  Table III : Side-by-side comparison (H&J original, V1, V3)

VERSION = 1  ->  Exact replication  (PWT 5.6, 1988, ICRG GADP, Sachs-Warner)
VERSION = 3  ->  Current cross-section (PWT 10.01, 2019, ICRG GADP, Fraser)
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

VERSION = 3   # change to 3 for current cross-section

OUT_DIR = "C:\\Users\\Adams\\OneDrive\\DE & E Research\\outputs\\"
MASTER  = OUT_DIR + f"merged_v{VERSION}.csv"
ALPHA   = 1/3

print(f"VERSION {VERSION} — reading {MASTER}")


VERSION 3 — reading C:\Users\Adams\OneDrive\DE & E Research\outputs\merged_v3.csv


In [72]:
# =============================================================================
# SECTION 1 — LOAD AND PREPARE DATA
# =============================================================================
print("=" * 65)
print(f"  Hall & Jones (1999) Replication — Version {VERSION}")
print("=" * 65)

df = pd.read_csv(MASTER, encoding='latin1')
print(f"\nLoaded {len(df)} countries")

df['mining_va'] = pd.to_numeric(df['mining_va'], errors='coerce').fillna(0.0)
df['gdp_nonmining_share'] = 1.0 - df['mining_va']
df['yl_adj'] = df['yl'] * df['gdp_nonmining_share']

for col in ['yl','ky_ratio','hc','social_infra',
            'distancefromeq','fr_trade','english_frac','we_lang_frac']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['log_yl']  = np.log(df['yl_adj'].clip(lower=1e-6))
df['cap_term'] = df['ky_ratio'].clip(lower=1e-6) ** (ALPHA / (1 - ALPHA))
df['hc_term'] = df['hc']
df['tfp']     = df['yl_adj'] / (df['cap_term'] * df['hc_term'])
df['log_si']  = np.log(df['social_infra'].clip(lower=1e-6))
df['log_tfp'] = np.log(df['tfp'].clip(lower=1e-6))

INSTRUMENTS = ['distancefromeq','fr_trade','english_frac','we_lang_frac']
INST_PREF   = ['distancefromeq','english_frac','we_lang_frac']

core_vars = ['yl_adj','ky_ratio','hc','social_infra'] + INSTRUMENTS
sA = df.dropna(subset=core_vars).copy().reset_index(drop=True)
print(f"Sample A (complete cases): {len(sA)} countries")
print(f"  Price base : {'1985 intl USD (PWT 5.6)' if VERSION==1 else '2017 intl USD (PWT 10.01)'}")
print(f"  Governance : {'ICRG GADP 1986-1995' if VERSION==1 else 'ICRG GADP 2010-2017'}")
print(f"  Education  : {'Barro-Lee 1985' if VERSION==1 else 'Barro-Lee 2015'}")
print(f"  Openness   : {'Sachs-Warner 1950-1992' if VERSION==1 else 'Fraser Institute 2015-2019'}")


  Hall & Jones (1999) Replication — Version 3

Loaded 177 countries
Sample A (complete cases): 99 countries
  Price base : 2017 intl USD (PWT 10.01)
  Governance : ICRG GADP 2010-2017
  Education  : Barro-Lee 2015
  Openness   : Fraser Institute 2015-2019


In [73]:
# ═══════════════════════════════════════════════════════════════════════════
# SECTION 2 — IMPUTATION (Hall & Jones method)
# ═══════════════════════════════════════════════════════════════════════════
print("\n--- Imputing missing social infrastructure ---")

# Hall & Jones impute missing governance/openness by regressing each component
# on the four instruments + quadratic distance from equator.
# We apply the same logic to social_infra directly (since we already
# combined governance and openness into one index).

def ols_impute(df_full, target_col, instrument_cols):
    """
    Regress target_col on instrument_cols (+ dist^2) using complete cases,
    then predict for all rows. Returns a new Series with imputed values.
    """
    # Add quadratic distance term
    regressors = instrument_cols + ['distancefromeq_sq']
    df_full = df_full.copy()
    df_full['distancefromeq_sq'] = df_full['distancefromeq'] ** 2

    # Training set: rows where both target and all regressors are observed
    train = df_full.dropna(subset=[target_col] + regressors)
    
    X_train = np.column_stack([np.ones(len(train))] +
                               [train[c].values for c in regressors])
    y_train = train[target_col].values

    # OLS: beta = (X'X)^{-1} X'y
    beta = np.linalg.lstsq(X_train, y_train, rcond=None)[0]

    # Predict for all rows that have complete instruments
    pred_col = target_col + '_imputed'
    df_full[pred_col] = np.nan
    has_inst = df_full.dropna(subset=regressors)
    X_pred = np.column_stack([np.ones(len(has_inst))] +
                               [has_inst[c].values for c in regressors])
    preds = X_pred @ beta
    # Clip predictions to [0, 1] (social infrastructure is bounded)
    preds = np.clip(preds, 0.0, 1.0)
    df_full.loc[has_inst.index, pred_col] = preds

    return df_full[pred_col], beta

# Impute social infrastructure
si_imputed, beta_si = ols_impute(df, 'social_infra', INSTRUMENTS)

# Build imputed series: use observed value if available, else imputed
df['social_infra_imp'] = df['social_infra'].combine_first(si_imputed)

n_imputed = df['social_infra'].isna().sum() - df['social_infra_imp'].isna().sum()
print(f"  Imputed social infrastructure for {n_imputed} additional countries")
print(f"  Imputation regression R²: {stats.pearsonr(df.dropna(subset=['social_infra','social_infra_imp'])['social_infra'], df.dropna(subset=['social_infra','social_infra_imp'])['social_infra_imp'])[0]**2:.3f}")

# ── Sample B: imputed sample ──────────────────────────────────────────────
core_vars_B = ['yl_adj', 'ky_ratio', 'hc', 'social_infra_imp',
               'distancefromeq', 'fr_trade', 'english_frac', 'we_lang_frac']
sB = df.dropna(subset=core_vars_B).copy().reset_index(drop=True)
sB['social_infra'] = sB['social_infra_imp']
sB['log_si']  = np.log(sB['social_infra'].clip(lower=1e-6))
sB['tfp']     = sB['yl_adj'] / (sB['cap_term'] * sB['hc_term'])
sB['log_tfp'] = np.log(sB['tfp'].clip(lower=1e-6))
sB['log_yl']  = np.log(sB['yl_adj'])
print(f"Sample B (with imputation): {len(sB)} countries")




--- Imputing missing social infrastructure ---
  Imputed social infrastructure for 21 additional countries
  Imputation regression R²: 1.000
Sample B (with imputation): 111 countries


In [74]:
# ═══════════════════════════════════════════════════════════════════════════
# SECTION 3 — ECONOMETRICS TOOLKIT (OLS and 2SLS from scratch)
# ═══════════════════════════════════════════════════════════════════════════

def ols(y, X):
    """
    OLS regression.
    Returns dict: beta, residuals, fitted, n, k, HC1 robust covariance matrix.
    X should include a constant column.
    """
    n, k = X.shape
    XX_inv = np.linalg.inv(X.T @ X)
    beta   = XX_inv @ X.T @ y
    yhat   = X @ beta
    e      = y - yhat

    # HC1 (heteroskedasticity-robust) covariance: (X'X)^{-1} X' diag(e²) X (X'X)^{-1} * n/(n-k)
    meat   = X.T @ np.diag(e**2) @ X
    V_hc1  = (n / (n - k)) * XX_inv @ meat @ XX_inv

    ss_res = e @ e
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2     = 1 - ss_res / ss_tot

    return {'beta': beta, 'e': e, 'yhat': yhat,
            'n': n, 'k': k, 'V': V_hc1, 'r2': r2}


def tsls(y, X_endog, X_exog, Z):
    """
    Two-Stage Least Squares (2SLS / IV).

    y       : outcome (n,)
    X_endog : endogenous regressors (n, p) — social infrastructure here
    X_exog  : included exogenous regressors (n, q) — just [1] (constant) here
    Z       : excluded instruments (n, m) — the four geographic/language vars

    Returns same dict as ols() plus first-stage F-stat and overid test.
    """
    n = len(y)

    # Full instrument matrix for first stage
    Z_full = np.column_stack([X_exog, Z])   # [const, instruments]
    X_full = np.column_stack([X_exog, X_endog])  # [const, S]

    # First stage: regress S on [const, instruments]
    fs = ols(X_endog.ravel(), Z_full)
    S_hat = fs['yhat'].reshape(-1, 1)

    # First-stage F-statistic (robust)
    # Test that all instrument coefficients are jointly zero
    # Positions of instrument coefficients: columns 1 onwards in Z_full
    n_inst = Z.shape[1]
    R = np.zeros((n_inst, Z_full.shape[1]))
    for i in range(n_inst):
        R[i, i + X_exog.shape[1]] = 1.0
    r = np.zeros(n_inst)
    Rb_r = R @ fs['beta'] - r
    V_fs  = fs['V']
    F_stat = (Rb_r @ np.linalg.inv(R @ V_fs @ R.T) @ Rb_r) / n_inst

    # Second stage: regress y on [const, S_hat]
    X2 = np.column_stack([X_exog, S_hat])
    ss = ols(y, X2)

    # Correct 2SLS standard errors:
    # Use actual residuals (y - X_full @ beta_2sls), not second-stage residuals
    beta_2sls = ss['beta']
    e_2sls    = y - X_full @ beta_2sls

    # HC1 robust covariance using actual residuals
    XX_inv_2 = np.linalg.inv(X2.T @ X2)
    meat_2   = X2.T @ np.diag(e_2sls**2) @ X2
    k2       = X2.shape[1]
    V_2sls   = (n / (n - k2)) * XX_inv_2 @ meat_2 @ XX_inv_2

    ss_res  = e_2sls @ e_2sls
    ss_tot  = ((y - y.mean())**2).sum()
    r2_2sls = 1 - ss_res / ss_tot

    # Sargan overidentification test (if more instruments than endogenous vars)
    # Regress 2SLS residuals on full instrument matrix; n*R² ~ chi²(m-p)
    overid_p = np.nan
    if n_inst > 1:
        sar = ols(e_2sls, Z_full)
        sargan_stat = n * sar['r2']
        df_sar = n_inst - X_endog.shape[1] if X_endog.ndim > 1 else n_inst - 1
        overid_p = 1 - stats.chi2.cdf(sargan_stat, df=df_sar)

    return {'beta': beta_2sls, 'e': e_2sls, 'yhat': ss['yhat'],
            'n': n, 'k': k2, 'V': V_2sls, 'r2': r2_2sls,
            'fs_beta': fs['beta'], 'fs_V': V_fs, 'F_first': F_stat,
            'overid_p': overid_p}


def fmt_result(res, labels):
    """Print a formatted regression result block."""
    beta = res['beta']
    V    = res['V']
    se   = np.sqrt(np.diag(V))
    t    = beta / se
    p    = 2 * (1 - stats.t.cdf(np.abs(t), df=res['n'] - res['k']))

    lines = []
    for i, lbl in enumerate(labels):
        stars = '***' if p[i] < 0.01 else ('**' if p[i] < 0.05 else
                ('*' if p[i] < 0.10 else ''))
        lines.append(f"  {lbl:<22} {beta[i]:>10.4f}  ({se[i]:.4f}) {stars}")
    lines.append(f"  {'N':<22} {res['n']:>10}")
    lines.append(f"  {'R²':<22} {res['r2']:>10.4f}")
    if 'F_first' in res:
        lines.append(f"  {'First-stage F':<22} {res['F_first']:>10.2f}")
    if 'overid_p' in res and not np.isnan(res['overid_p']):
        lines.append(f"  {'Overid test p-val':<22} {res['overid_p']:>10.3f}")
    return '\n'.join(lines)



In [75]:
# ═══════════════════════════════════════════════════════════════════════════
# SECTION 4 — TABLE I: LEVELS ACCOUNTING
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  TABLE I — Levels Accounting Decomposition")
print("=" * 65)
print("  Methodology: Y/L = (K/Y)^(α/(1-α)) × h × A,  α = 1/3")
print("  All values relative to the United States = 1.00")
print()

def levels_accounting(sample, label):
    s = sample.copy()

    # Normalise everything to US = 1
    usa = s[s['iso3'] == 'USA'].iloc[0]
    s['rel_yl']  = s['yl_adj']   / usa['yl_adj']
    s['rel_cap'] = s['cap_term'] / usa['cap_term']
    s['rel_hc']  = s['hc_term']  / usa['hc_term']
    s['rel_tfp'] = s['tfp']      / usa['tfp']

    # Variance decomposition (in logs, following Hall & Jones eq. 6)
    # var(log Y/L) = cov(log Y/L, log cap) + cov(log Y/L, log h) + cov(log Y/L, log A)
    log_yl  = np.log(s['rel_yl'].clip(1e-6))
    log_cap = np.log(s['rel_cap'].clip(1e-6))
    log_hc  = np.log(s['rel_hc'].clip(1e-6))
    log_tfp_r = np.log(s['rel_tfp'].clip(1e-6))

    var_yl  = np.var(log_yl)
    cov_cap = np.cov(log_yl, log_cap)[0, 1]
    cov_hc  = np.cov(log_yl, log_hc)[0, 1]
    cov_tfp = np.cov(log_yl, log_tfp_r)[0, 1]

    share_cap = cov_cap / var_yl
    share_hc  = cov_hc  / var_yl
    share_tfp = cov_tfp / var_yl

    # Range: ratio of 90th to 10th percentile
    p90_p10 = lambda x: np.percentile(x, 90) / np.percentile(x, 10)

    print(f"\n  [{label}]  N = {len(s)}")
    print(f"\n  {'Country':<25} {'Y/L':>8} {'Cap term':>10} {'h':>8} {'TFP (A)':>10}  {'S':>8}")
    print(f"  {'-'*70}")

    # Show 15 representative countries (sorted by Y/L)
    show = s.sort_values('rel_yl', ascending=False)
    display_isos = ['USA','CHE','CAN','AUS','GBR','JPN','FRA','DEU',
                    'BRA','COL','THA','CHN','IND','KEN','NGA','NER']
    show_rows = show[show['iso3'].isin(display_isos)]
    for _, row in show_rows.iterrows():
        si_val = f"{row['social_infra']:.3f}" if not pd.isna(row['social_infra']) else '  NA '
        print(f"  {row['country']:<25} {row['rel_yl']:>8.3f} {row['rel_cap']:>10.3f} "
              f"{row['rel_hc']:>8.3f} {row['rel_tfp']:>10.3f}  {si_val:>8}")

    print(f"\n  90th/10th percentile ratios:")
    print(f"    Y/L:      {p90_p10(s['rel_yl']):.1f}x")
    print(f"    Cap term: {p90_p10(s['rel_cap']):.1f}x")
    print(f"    h:        {p90_p10(s['rel_hc']):.1f}x")
    print(f"    TFP (A):  {p90_p10(s['rel_tfp']):.1f}x")

    print(f"\n  Variance decomposition (fraction of var(log Y/L) explained):")
    print(f"    Capital term: {share_cap:.3f}")
    print(f"    Education h:  {share_hc:.3f}")
    print(f"    TFP (A):      {share_tfp:.3f}")
    print(f"    Sum:          {share_cap + share_hc + share_tfp:.3f}")

    return s  # return enriched sample with rel_ columns

sA_enriched = levels_accounting(sA, "Sample A — complete cases, no imputation")
sB_enriched = levels_accounting(sB, "Sample B — with imputation")




  TABLE I — Levels Accounting Decomposition
  Methodology: Y/L = (K/Y)^(α/(1-α)) × h × A,  α = 1/3
  All values relative to the United States = 1.00


  [Sample A — complete cases, no imputation]  N = 99

  Country                        Y/L   Cap term        h    TFP (A)         S
  ----------------------------------------------------------------------
  Switzerland                  1.008      1.115    0.984      0.918     0.828
  United States                1.000      1.000    1.000      1.000     0.852
  France                       0.807      1.070    0.879      0.858     0.736
  Australia                    0.765      1.015    0.975      0.773     0.842
  Germany                      0.745      1.014    0.944      0.778     0.811
  Canada                       0.719      1.044    0.967      0.713     0.847
  United Kingdom               0.709      1.010    0.944      0.743     0.817
  Japan                        0.562      1.177    0.941      0.508     0.791
  Brazil           

In [76]:
# =============================================================================
# SECTION 5 — TABLE II: IV REGRESSIONS
# =============================================================================
print("\n" + "=" * 65)
print("  TABLE II — OLS and 2SLS Regressions")
print("  Dependent variable: log(Y/L)")
print("  Robust (HC1) standard errors in parentheses")
print("=" * 65)

def run_regressions(sample, label):
    s  = sample.copy().dropna(subset=['log_yl','social_infra']+INSTRUMENTS)
    n  = len(s)
    y  = s['log_yl'].values
    S  = s['social_infra'].values.reshape(-1,1)
    X1 = np.ones((n,1))

    res_ols  = ols(y, np.column_stack([X1,S]))
    res_iv   = tsls(y, S, X1, s[INSTRUMENTS].values)
    res_ivp  = tsls(y, S, X1, s[INST_PREF].values)

    print(f"\n  [{label}]  N = {n}")
    print(f"\n  OLS:")
    print(fmt_result(res_ols, ['Constant','Social infra (S)']))
    print(f"\n  2SLS (all 4 instruments):")
    print(fmt_result(res_iv,  ['Constant','Social infra (S)']))
    print(f"\n  2SLS preferred (3 instruments: dist_eq, English, WE lang):")
    print(fmt_result(res_ivp, ['Constant','Social infra (S)']))

    print(f"\n  Sensitivity — instrument subsets:")
    subsets = [
        (['distancefromeq'],     "dist_eq only"),
        (['distancefromeq','fr_trade'],  "dist + FR trade"),
        (INST_PREF,              "dist + language  [preferred]"),
        (INSTRUMENTS,            "all 4 instruments"),
    ]
    print(f"  {'Instruments':<38} {'beta(S)':>7}  {'SE':>7}  {'F-stat':>7}  {'Overid p':>9}")
    print(f"  {'-'*75}")
    for inst_list, desc in subsets:
        r   = tsls(y, S, X1, s[inst_list].values)
        se  = np.sqrt(r['V'][1,1])
        oid = f"{r['overid_p']:.3f}" if not np.isnan(r['overid_p']) else "  n/a"
        print(f"  {desc:<38} {r['beta'][1]:>7.3f}  {se:>7.4f}  {r['F_first']:>7.2f}  {oid:>9}")

    return res_ols, res_iv, res_ivp, s

ols_A, iv_A, ivp_A, sA_reg = run_regressions(sA, "Sample A — complete cases")

try:
    ols_B, iv_B, ivp_B, sB_reg = run_regressions(sB, "Sample B — with imputation")
except NameError:
    print("\n  (Sample B not available — run Cell 2 first)")
    sB_reg = None

print("\n" + "=" * 65)
print("  OLS vs 2SLS summary")
print("=" * 65)
print(f"  {'Sample':<28} {'OLS':>7}  {'2SLS (4-inst)':>14}  {'2SLS (3-inst)':>14}")
print(f"  {'-'*68}")
rows = [("A — complete cases", ols_A, iv_A, ivp_A)]
if sB_reg is not None:
    rows.append(("B — with imputation", ols_B, iv_B, ivp_B))
for lbl, ro, ri, rip in rows:
    print(f"  {lbl:<28} {ro['beta'][1]:>7.3f}  {ri['beta'][1]:>14.3f}  {rip['beta'][1]:>14.3f}")
print()
print("  H&J original: OLS=3.29  2SLS=5.14")
print("  V1 overid failure (p=0.037) with all 4 instruments.")
print("  Preferred 3-instrument spec passes overid — result unchanged.")



  TABLE II — OLS and 2SLS Regressions
  Dependent variable: log(Y/L)
  Robust (HC1) standard errors in parentheses

  [Sample A — complete cases]  N = 99

  OLS:
  Constant                   6.2810  (0.4218) ***
  Social infra (S)           6.5363  (0.6026) ***
  N                              99
  R²                         0.5860

  2SLS (all 4 instruments):
  Constant                   5.2028  (0.5136) ***
  Social infra (S)           8.2721  (0.7869) ***
  N                              99
  R²                         0.5447
  First-stage F               29.50
  Overid test p-val           0.203

  2SLS preferred (3 instruments: dist_eq, English, WE lang):
  Constant                   5.2673  (0.5261) ***
  Social infra (S)           8.1682  (0.8125) ***
  N                              99
  R²                         0.5495
  First-stage F               42.73
  Overid test p-val           0.110

  Sensitivity — instrument subsets:
  Instruments                            beta(S) 

  all 4 instruments                        8.415   0.8544    37.68      0.031

  OLS vs 2SLS summary
  Sample                           OLS   2SLS (4-inst)   2SLS (3-inst)
  --------------------------------------------------------------------
  A — complete cases             6.536           8.272           8.168
  B — with imputation            6.603           8.415           8.480

  H&J original: OLS=3.29  2SLS=5.14
  V1 overid failure (p=0.037) with all 4 instruments.
  Preferred 3-instrument spec passes overid — result unchanged.


In [77]:
# ═══════════════════════════════════════════════════════════════════════════
# SECTION 6 — FIGURE I: Y/L vs Social Infrastructure
# ═══════════════════════════════════════════════════════════════════════════
print("\n--- Generating Figure I ---")

# Highlight a selection of countries with labels
LABEL_ISOS = {
    'USA', 'CHE', 'NOR', 'JPN', 'GBR', 'FRA', 'DEU', 'CAN', 'AUS',
    'BRA', 'MEX', 'ARG', 'COL', 'THA', 'MYS', 'KOR',
    'CHN', 'IND', 'PAK', 'BGD', 'NGA', 'KEN', 'ETH', 'NER', 'ZAF',
    'EGY', 'GHA', 'TZA', 'UGA', 'SEN',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Figure I — Output per Worker vs. Social Infrastructure",
             fontsize=13, fontweight='bold', y=1.01)

for ax, sample, title in [
    (axes[0], sA_reg, "Sample A: Complete cases (N={})".format(len(sA_reg))),
    (axes[1], sB_reg, "Sample B: With imputation (N={})".format(len(sB_reg))),
]:
    s = sample.dropna(subset=['log_yl', 'social_infra'])

    ax.scatter(s['social_infra'], s['log_yl'],
               color='steelblue', alpha=0.55, s=30, zorder=2, linewidths=0)

    # Fitted OLS line
    xi = np.linspace(s['social_infra'].min(), s['social_infra'].max(), 200)
    X_fit = np.column_stack([np.ones(200), xi])
    y_fit = X_fit @ ols(s['log_yl'].values,
                        np.column_stack([np.ones(len(s)), s['social_infra'].values]))['beta']
    ax.plot(xi, y_fit, color='firebrick', linewidth=1.8, zorder=3, label='OLS fit')

    # Country labels
    for _, row in s.iterrows():
        if row['iso3'] in LABEL_ISOS:
            ax.annotate(row['iso3'],
                        xy=(row['social_infra'], row['log_yl']),
                        xytext=(3, 1), textcoords='offset points',
                        fontsize=6.5, color='#333333', zorder=4)

    ax.set_xlabel("Social Infrastructure (S)", fontsize=11)
    ax.set_ylabel("log(Output per Worker)", fontsize=11)
    ax.set_title(title, fontsize=10, pad=6)
    ax.set_xlim(-0.02, 1.05)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Annotate with IV coefficient
    iv_res = iv_A if sample is sA_reg else iv_B
    b  = iv_res['beta'][1]
    se = np.sqrt(iv_res['V'][1, 1])
    ax.text(0.03, 0.96, f"2SLS β = {b:.2f} ({se:.3f})",
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#cccccc', alpha=0.9))

plt.tight_layout()
fig1_path = OUT_DIR + 'figure_I_yl_vs_si.png'
plt.savefig(fig1_path, dpi=160, bbox_inches='tight')
plt.close()
print(f"  Saved to {fig1_path}")




--- Generating Figure I ---
  Saved to C:\Users\Adams\OneDrive\DE & E Research\outputs\figure_I_yl_vs_si.png


In [78]:
# ═══════════════════════════════════════════════════════════════════════════
# SECTION 7 — FIGURE II: TFP vs Social Infrastructure
# ═══════════════════════════════════════════════════════════════════════════
print("--- Generating Figure II ---")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Figure II — TFP (Productivity) vs. Social Infrastructure",
             fontsize=13, fontweight='bold', y=1.01)

for ax, sample, title in [
    (axes[0], sA_enriched, "Sample A: Complete cases (N={})".format(len(sA_enriched.dropna(subset=['log_tfp','social_infra'])))),
    (axes[1], sB_enriched, "Sample B: With imputation (N={})".format(len(sB_enriched.dropna(subset=['log_tfp','social_infra'])))),
]:
    s = sample.dropna(subset=['log_tfp', 'social_infra']).copy()
    s = s[np.isfinite(s['log_tfp']) & np.isfinite(s['social_infra'])]

    ax.scatter(s['social_infra'], s['log_tfp'],
               color='darkorange', alpha=0.55, s=30, zorder=2, linewidths=0)

    # Fitted OLS line
    xi = np.linspace(s['social_infra'].min(), s['social_infra'].max(), 200)
    X_fit = np.column_stack([np.ones(200), xi])
    y_fit = X_fit @ ols(s['log_tfp'].values,
                        np.column_stack([np.ones(len(s)), s['social_infra'].values]))['beta']
    ax.plot(xi, y_fit, color='firebrick', linewidth=1.8, zorder=3)

    # Country labels
    for _, row in s.iterrows():
        if row['iso3'] in LABEL_ISOS:
            ax.annotate(row['iso3'],
                        xy=(row['social_infra'], row['log_tfp']),
                        xytext=(3, 1), textcoords='offset points',
                        fontsize=6.5, color='#333333', zorder=4)

    # Correlation
    r, p_val = stats.pearsonr(s['social_infra'], s['log_tfp'])
    ax.text(0.03, 0.96, f"r = {r:.3f}",
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#cccccc', alpha=0.9))

    ax.set_xlabel("Social Infrastructure (S)", fontsize=11)
    ax.set_ylabel("log(TFP)", fontsize=11)
    ax.set_title(title, fontsize=10, pad=6)
    ax.set_xlim(-0.02, 1.05)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig2_path = OUT_DIR + 'figure_II_tfp_vs_si.png'
plt.savefig(fig2_path, dpi=160, bbox_inches='tight')
plt.close()
print(f"  Saved to {fig2_path}")



--- Generating Figure II ---
  Saved to C:\Users\Adams\OneDrive\DE & E Research\outputs\figure_II_tfp_vs_si.png


In [79]:
# =============================================================================
# SECTION 8 — DOCUMENTED DEVIATIONS
# =============================================================================
print("\n" + "=" * 65)
print(f"  DEVIATIONS FROM HALL & JONES (1999) — Version {VERSION}")
print("=" * 65)

if VERSION == 1:
    deviations = [
        ("Sample size",
         "H&J N=127. Ours: 98 complete cases / 109 with imputation.\n"
         "    Cause: ICRG did not cover ~26 small countries in 1986-1995.\n"
         "    Key result unchanged: 2SLS beta = 5.09 vs H&J 5.14."),
        ("USA/Niger Y/L gap",
         "H&J report 35x. We find 33x.\n"
         "    Minor PWT 5.6 data revisions since 1999 publication."),
        ("Capital stock",
         "~40 countries have inconsistent PWT 5.6 KAPW units.\n"
         "    Fix: perpetual inventory on PWT 5.6 investment shares (ci)."),
        ("Openness coverage",
         "H&J use Sachs-Warner 1950-1994. Our data ends 1992 (2-year gap).\n"
         "    Effect: Negligible — 2 years out of 40+ in the fraction."),
        ("FR trade instrument",
         "Sargan overid p=0.037 with all 4 instruments.\n"
         "    Preferred 3-instrument spec: beta=5.06, overid p=0.183.\n"
         "    FR trade likely has direct income effects beyond institutions."),
        ("Standard errors",
         "H&J use bootstrap (10,000 reps). We use HC1 robust SEs."),
    ]
else:
    deviations = [
        ("Cross-section year",
         "H&J use 1988. We use 2019 (most recent pre-COVID year)."),
        ("Price base",
         "H&J PWT 5.6 (1985 prices). We use PWT 10.01 (2017 prices)."),
        ("Governance window",
         "H&J ICRG GADP 1986-1995. We use ICRG GADP 2010-2017.\n"
         "    Same source and formula — window shifted to match 2019 cross-section."),
        ("Education",
         "H&J Barro-Lee 1985, age 25+. We use Barro-Lee 2015, age 25-64."),
        ("Openness",
         "H&J Sachs-Warner (binary). We use Fraser Institute Area 5\n"
         "    (continuous, avg 2015-2019, normalised 0-1)."),
        ("USA/Niger Y/L gap",
         "H&J 35x. We find 40.9x — gap has widened since 1988."),
        ("2SLS beta",
         "H&J beta=5.14. We find 8.27 in 2019.\n"
         "    The institutional premium on output has strengthened over 30 years."),
        ("Standard errors",
         "H&J use bootstrap. We use HC1 robust SEs."),
    ]

for title, desc in deviations:
    print(f"\n  [{title}]")
    print(f"    {desc}")

print(f"\n  Figures saved to: {OUT_DIR}")



  DEVIATIONS FROM HALL & JONES (1999) — Version 3

  [Cross-section year]
    H&J use 1988. We use 2019 (most recent pre-COVID year).

  [Price base]
    H&J PWT 5.6 (1985 prices). We use PWT 10.01 (2017 prices).

  [Governance window]
    H&J ICRG GADP 1986-1995. We use ICRG GADP 2010-2017.
    Same source and formula — window shifted to match 2019 cross-section.

  [Education]
    H&J Barro-Lee 1985, age 25+. We use Barro-Lee 2015, age 25-64.

  [Openness]
    H&J Sachs-Warner (binary). We use Fraser Institute Area 5
    (continuous, avg 2015-2019, normalised 0-1).

  [USA/Niger Y/L gap]
    H&J 35x. We find 40.9x — gap has widened since 1988.

  [2SLS beta]
    H&J beta=5.14. We find 8.27 in 2019.
    The institutional premium on output has strengthened over 30 years.

  [Standard errors]
    H&J use bootstrap. We use HC1 robust SEs.

  Figures saved to: C:\Users\Adams\OneDrive\DE & E Research\outputs\


In [80]:
# =============================================================================
# SECTION 9 — TABLE III: SIDE-BY-SIDE COMPARISON
# Self-contained — reads both CSVs directly, no other cells required.
# =============================================================================

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

OUT_DIR     = "C:\\Users\\Adams\\OneDrive\\DE & E Research\\outputs\\"
V1_FILE     = OUT_DIR + "merged_v1.csv"
V3_FILE     = OUT_DIR + "merged_v3.csv"
ALPHA       = 1/3
INSTRUMENTS = ['distancefromeq','fr_trade','english_frac','we_lang_frac']
INST_PREF   = ['distancefromeq','english_frac','we_lang_frac']

def _ols(y, X):
    n, k = X.shape
    XXi  = np.linalg.inv(X.T @ X)
    beta = XXi @ X.T @ y
    e    = y - X @ beta
    V    = (n/(n-k)) * XXi @ (X.T @ np.diag(e**2) @ X) @ XXi
    r2   = 1 - (e@e)/((y-y.mean())**2).sum()
    return dict(beta=beta, e=e, V=V, r2=r2, n=n, k=k)

def _tsls(y, S, X1, Z):
    n    = len(y)
    Zf   = np.column_stack([X1, Z])
    Xf   = np.column_stack([X1, S])
    fs   = _ols(S.ravel(), Zf)
    Sh   = (Zf @ fs['beta']).reshape(-1,1)
    ss   = _ols(y, np.column_stack([X1, Sh]))
    beta = ss['beta']
    e    = y - Xf @ beta
    XXi2 = np.linalg.inv(np.column_stack([X1,Sh]).T @ np.column_stack([X1,Sh]))
    V    = (n/(n-2)) * XXi2 @ (np.column_stack([X1,Sh]).T @ np.diag(e**2) @ np.column_stack([X1,Sh])) @ XXi2
    r2   = 1 - (e@e)/((y-y.mean())**2).sum()
    ni   = Z.shape[1]
    R    = np.zeros((ni, Zf.shape[1]))
    for i in range(ni): R[i, i+X1.shape[1]] = 1.0
    Rb   = R @ fs['beta']
    F    = (Rb @ np.linalg.inv(R @ fs['V'] @ R.T) @ Rb) / ni
    oid  = np.nan
    if ni > 1:
        sar = _ols(e, Zf)
        oid = 1 - stats.chi2.cdf(n * sar['r2'], df=ni-1)
    return dict(beta=beta, V=V, r2=r2, n=n, F_first=F, overid_p=oid)

def _prep(path):
    df = pd.read_csv(path, encoding='latin1')
    for c in ['yl','ky_ratio','hc','social_infra',
              'distancefromeq','fr_trade','english_frac','we_lang_frac','mining_va']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['mining_va'] = df['mining_va'].fillna(0.0)
    df['yl_adj']   = df['yl'] * (1 - df['mining_va'])
    df['cap_term'] = df['ky_ratio'].clip(lower=1e-6) ** (ALPHA/(1-ALPHA))
    df['hc_term']  = df['hc']
    df['tfp']      = df['yl_adj'] / (df['cap_term'] * df['hc_term'])
    df['log_yl']   = np.log(df['yl_adj'].clip(lower=1e-6))
    sA = df.dropna(subset=['yl_adj','ky_ratio','hc','social_infra']+INSTRUMENTS)
    # Drop K/Y outliers beyond Q3 + 3*IQR (e.g. Venezuela 2019: K/Y=127
    # due to economic collapse — output halved while oil capital stock remained)
    q1, q3 = sA['ky_ratio'].quantile(0.25), sA['ky_ratio'].quantile(0.75)
    sA = sA[sA['ky_ratio'] <= q3 + 3*(q3-q1)].copy()
    return df, sA.reset_index(drop=True)

def _impute(df, sA, instruments):
    """
    Impute missing social_infra for countries that have all instruments
    but missing governance or openness. Uses OLS on instruments + dist^2.
    Returns sB: sample with observed + imputed social_infra.
    """
    df = df.copy()
    df['dist_sq'] = df['distancefromeq'] ** 2
    regressors = instruments + ['dist_sq']

    # Training set: observed social_infra + complete instruments
    train = df.dropna(subset=['social_infra'] + regressors)
    X_tr  = np.column_stack([np.ones(len(train))] +
                             [train[c].values for c in regressors])
    y_tr  = train['social_infra'].values
    beta  = np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]

    # Predict for all rows with complete instruments
    has_inst = df.dropna(subset=regressors +
                         ['yl_adj','ky_ratio','hc'] + instruments)
    X_pr  = np.column_stack([np.ones(len(has_inst))] +
                             [has_inst[c].values for c in regressors])
    preds = np.clip(X_pr @ beta, 0.0, 1.0)
    df.loc[has_inst.index, 'si_imp'] = preds

    # Use observed where available, else imputed
    df['social_infra_B'] = df['social_infra'].combine_first(df['si_imp'])

    # Build Sample B
    core_B = ['yl_adj','ky_ratio','hc','social_infra_B'] + instruments
    sB = df.dropna(subset=core_B).copy().reset_index(drop=True)
    sB['social_infra'] = sB['social_infra_B']
    sB['log_yl']  = np.log(sB['yl_adj'].clip(lower=1e-6))
    sB['cap_term'] = sB['ky_ratio'].clip(lower=1e-6) ** (ALPHA/(1-ALPHA))
    sB['hc_term'] = sB['hc']
    sB['tfp']     = sB['yl_adj'] / (sB['cap_term'] * sB['hc_term'])

    n_imp = df['social_infra'].isna().sum() - df['social_infra_B'].isna().sum()
    return sB, n_imp

def _levels(sA):
    usa  = sA[sA['iso3']=='USA'].iloc[0]
    lyl  = np.log((sA['yl_adj']/usa['yl_adj']).clip(1e-6))
    lcap = np.log((sA['cap_term']/usa['cap_term']).clip(1e-6))
    lhc  = np.log((sA['hc_term']/usa['hc_term']).clip(1e-6))
    ltfp = np.log((sA['tfp']/usa['tfp']).clip(1e-6))
    v    = np.var(lyl)
    ner  = sA[sA['iso3']=='NER']
    gap  = usa['yl_adj']/ner['yl_adj'].iloc[0] if len(ner) else np.nan
    return (np.cov(lyl,lcap)[0,1]/v, np.cov(lyl,lhc)[0,1]/v,
            np.cov(lyl,ltfp)[0,1]/v, gap)

def _reg(sA, inst):
    s  = sA.dropna(subset=['log_yl','social_infra']+inst)
    y  = s['log_yl'].values
    S  = s['social_infra'].values.reshape(-1,1)
    X1 = np.ones((len(s),1))
    o  = _ols(y, np.column_stack([X1,S]))
    iv = _tsls(y, S, X1, s[inst].values)
    return (o['beta'][1],  np.sqrt(o['V'][1,1]),
            iv['beta'][1], np.sqrt(iv['V'][1,1]),
            iv['F_first'], iv['overid_p'], len(s))

# ── Compute ──────────────────────────────────────────────────────────────────
print("  Computing...")
df1, sA1 = _prep(V1_FILE); df3, sA3 = _prep(V3_FILE)
c1,h1,t1,gap1 = _levels(sA1); c3,h3,t3,gap3 = _levels(sA3)
sB1, n_imp1  = _impute(df1, sA1, INSTRUMENTS)
sB3, n_imp3  = _impute(df3, sA3, INSTRUMENTS)
c1b,h1b,t1b,gap1b = _levels(sB1); c3b,h3b,t3b,gap3b = _levels(sB3)
b_ols1,se_ols1,b_iv1,se_iv1,F1,oid1,n1 = _reg(sA1, INSTRUMENTS)
b_ols3,se_ols3,b_iv3,se_iv3,F3,oid3,n3 = _reg(sA3, INSTRUMENTS)
_,_,b_iv1p,se_iv1p,F1p,oid1p,_  = _reg(sA1, INST_PREF)
_,_,b_iv3p,se_iv3p,F3p,oid3p,_  = _reg(sA3, INST_PREF)
b_ols1b,se_ols1b,b_iv1b,se_iv1b,F1b,oid1b,n1b = _reg(sB1, INSTRUMENTS)
b_ols3b,se_ols3b,b_iv3b,se_iv3b,F3b,oid3b,n3b = _reg(sB3, INSTRUMENTS)
_,_,b_iv1pb,se_iv1pb,F1pb,oid1pb,_ = _reg(sB1, INST_PREF)
_,_,b_iv3pb,se_iv3pb,F3pb,oid3pb,_ = _reg(sB3, INST_PREF)

# H&J originals
HJ = dict(n=127, gap=35.0, cap=0.228, hc=0.143, tfp=0.601,
          bols=3.29, seols=0.398, biv=5.14, seiv=0.508, oid=0.256)

# ── Print ─────────────────────────────────────────────────────────────────────
W = 70
def r(lbl, a, b, c, f=None):
    fa = (f.format(a) if f else str(a)) if not isinstance(a,str) else a
    fb = (f.format(b) if f else str(b)) if not isinstance(b,str) else b
    fc = (f.format(c) if f else str(c)) if not isinstance(c,str) else c
    print(f"  {lbl:<34} {fa:>10}  {fb:>10}  {fc:>10}")
def se(a,b,c):
    print(f"  {'':<34} {'('+f'{a:.3f}'+ ')':>10}  {'('+f'{b:.3f}'+')':>10}  {'('+f'{c:.3f}'+')':>10}")

print()
print("=" * W)
print("  TABLE III — REPLICATION COMPARISON")
print("  Hall & Jones (1999)  |  Version 1: 1988  |  Version 3: 2019")
print("=" * W)
print(f"  {'':<34} {'H&J (1999)':>10}  {'V1: 1988':>10}  {'V3: 2019':>10}")
print(f"  {'-'*W}")

print(f"\n  PANEL A — DATA SOURCES")
r("Output & capital",    "PWT 5.6",   "PWT 5.6",   "PWT 10.01")
r("Price base",          "1985 USD",  "1985 USD",  "2017 USD")
r("Governance source",   "ICRG GADP", "ICRG GADP", "ICRG GADP")
r("Governance window",   "1986-1995", "1986-1995", "2010-2017")
r("Education",           "BL 1985",   "BL 1985",   "BL 2015")
r("Openness",            "Sachs-W.",  "Sachs-W.",  "Fraser")

print(f"\n  PANEL B — LEVELS ACCOUNTING (Sample A, complete cases)")
r("N (Sample A)",         HJ['n'],    n1,   n3,   f='{:>10d}')
r("N (Sample B imputed)",  "—",        n1b,  n3b,  f='{:>10d}')
print(f"  (imputed {n_imp1} V1 / {n_imp3} V3 countries)")
r("USA / Niger Y/L gap", f"{HJ['gap']:.1f}x", f"{gap1:.1f}x", f"{gap3:.1f}x")
r("Capital term share",  HJ['cap'],  c1,   c3,   f='{:>10.3f}')
r("Human capital share", HJ['hc'],   h1,   h3,   f='{:>10.3f}')
r("TFP share",           HJ['tfp'],  t1,   t3,   f='{:>10.3f}')
r("Sum of shares",       f"{sum(HJ[v] for v in ['cap','hc','tfp']):.3f}",
                          f"{c1+h1+t1:.3f}", f"{c3+h3+t3:.3f}")

print(f"\n  PANEL C — REGRESSIONS: all 4 instruments")
r("OLS beta (S)",        HJ['bols'],  b_ols1, b_ols3, f='{:>10.3f}')
se(HJ['seols'], se_ols1, se_ols3)
r("2SLS beta (S)",       HJ['biv'],   b_iv1,  b_iv3,  f='{:>10.3f}')
se(HJ['seiv'], se_iv1, se_iv3)
r("First-stage F",       "—",          f"{F1:.2f}",  f"{F3:.2f}")
r("Overid p-value",      HJ['oid'],   oid1,   oid3,   f='{:>10.3f}')
print(f"\n  Sample B (with imputation):")
r("  OLS beta (S)",        "—",         b_ols1b, b_ols3b, f='{:>10.3f}')
print(f"  {'  SE':<34} {'—':>10}  {'('+f'{se_ols1b:.3f}'+')':>10}  {'('+f'{se_ols3b:.3f}'+')':>10}")
r("  2SLS beta (S)",       "—",         b_iv1b,  b_iv3b,  f='{:>10.3f}')
print(f"  {'  SE':<34} {'—':>10}  {'('+f'{se_iv1b:.3f}'+')':>10}  {'('+f'{se_iv3b:.3f}'+')':>10}")
r("  First-stage F",       "—",         f"{F1b:.2f}", f"{F3b:.2f}")
r("  Overid p-value",      "—",         oid1b,   oid3b,   f='{:>10.3f}')

print(f"\n  PANEL D — PREFERRED SPEC: dist_eq + English + WE lang (3 inst.)")
r("2SLS beta (S)",       "—",    b_iv1p,  b_iv3p,  f='{:>10.3f}')
print(f"  {'  SE':<34} {'—':>10}  {'('+f'{se_iv1p:.3f}'+')':>10}  {'('+f'{se_iv3p:.3f}'+')':>10}")
r("First-stage F",       "—",    f"{F1p:.2f}", f"{F3p:.2f}")
r("Overid p-value",      "—",    oid1p,   oid3p,   f='{:>10.3f}')
print(f"\n  Sample B (with imputation):")
r("  2SLS beta (S)",       "—",    b_iv1pb,  b_iv3pb,  f='{:>10.3f}')
print(f"  {'  SE':<34} {'—':>10}  {'('+f'{se_iv1pb:.3f}'+')':>10}  {'('+f'{se_iv3pb:.3f}'+')':>10}")
r("  First-stage F",       "—",    f"{F1pb:.2f}", f"{F3pb:.2f}")
r("  Overid p-value",      "—",    oid1pb,  oid3pb,   f='{:>10.3f}')

print(f"\n  PANEL E — KEY FINDINGS")
r("IV / OLS ratio",      "1.56x", f"{b_iv1/b_ols1:.2f}x", f"{b_iv3/b_ols3:.2f}x")
chg = (b_iv3 - b_iv1) / b_iv1 * 100
r("2SLS beta change V1->V3", "—", "baseline", f"+{chg:.0f}%")
r("Y/L gap change",      "—",    "35x -> 33x", "35x -> 41x")

print(f"\n  {'='*W}")
print(f"  HC1 robust SEs in parentheses. H&J use bootstrap SEs.")
print(f"  V1 Panel C overid failure (p=0.037) driven by FR trade instrument.")
print(f"  Panel D drops FR trade: result unchanged, overid passes.")
print(f"  BL = Barro-Lee. Sachs-W. = Sachs-Warner. Fraser = Fraser Inst. Area 5.")
print(f"  {'='*W}")


  Computing...

  TABLE III — REPLICATION COMPARISON
  Hall & Jones (1999)  |  Version 1: 1988  |  Version 3: 2019
                                     H&J (1999)    V1: 1988    V3: 2019
  ----------------------------------------------------------------------

  PANEL A — DATA SOURCES
  Output & capital                      PWT 5.6     PWT 5.6   PWT 10.01
  Price base                           1985 USD    1985 USD    2017 USD
  Governance source                   ICRG GADP   ICRG GADP   ICRG GADP
  Governance window                   1986-1995   1986-1995   2010-2017
  Education                             BL 1985     BL 1985     BL 2015
  Openness                             Sachs-W.    Sachs-W.      Fraser

  PANEL B — LEVELS ACCOUNTING (Sample A, complete cases)
  N (Sample A)                              127          98          98
  N (Sample B imputed)                        —         109         111
  (imputed 11 V1 / 12 V3 countries)
  USA / Niger Y/L gap                     35